# Structure of the raw SFTR securities lending data

Source table `crp_sftds_ecb.trade_states_securitieslending`. Each section asks one question about the raw data, runs the query that answers it and follows up on what the answer leaves open.

In [22]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

## 2. What is one row?

Follow one trade through the table. `tec_ruti` identifies a trade across both reporting sides.

In [23]:
query = f"""

SELECT reference_period, reporting_cpty_id, other_cpty_id, counterparty_side, uti
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti = '549300GWM9UJLZCGSN04R0MUWSFPU8MPRO8K5P83F2KUVLIFOYYYLY7A86QTJNMN76CPND8WN28Z'
ORDER BY reference_period

"""
df = pd.read_sql_query(query, cnxn)
df['reference_period'].value_counts()

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\108234014.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


reference_period
2026-06-04    2
2026-07-28    2
2026-07-14    2
2026-07-15    2
2026-07-16    2
2026-07-17    2
2026-07-20    2
2026-07-21    2
2026-07-22    2
2026-07-23    2
2026-07-24    2
2026-07-27    2
2026-07-29    2
2026-07-10    2
2026-07-30    2
2026-07-31    2
2026-08-03    2
2026-08-04    2
2026-08-05    2
2026-08-06    2
2026-08-07    2
2026-08-10    2
2026-08-11    2
2026-08-12    2
2026-07-13    2
2026-07-09    2
2026-06-05    2
2026-06-22    2
2026-06-08    2
2026-06-09    2
2026-06-10    2
2026-06-11    2
2026-06-12    2
2026-06-15    2
2026-06-16    2
2026-06-17    2
2026-06-18    2
2026-06-19    2
2026-06-23    2
2026-07-08    2
2026-06-24    2
2026-06-25    2
2026-06-26    2
2026-06-29    2
2026-06-30    2
2026-07-01    2
2026-07-02    2
2026-07-03    2
2026-07-06    2
2026-07-07    2
2026-08-13    2
Name: count, dtype: int64

Answer. One row per reporting side and day, so a trade reported by both sides gives two rows per day. The two legs on one day share the UTI, with reporting and other counterparty swapped and opposite `counterparty_side`.

In [24]:
df.head(2)

,reference_period,reporting_cpty_id,other_cpty_id,counterparty_side,uti
0,2026-06-04,R0MUWSFPU8MPRO8K5P83,549300GWM9UJLZCGSN04,GIVE,F2KUVLIFOYYYLY7A86QTJNMN76CPND8WN28Z
1,2026-06-04,549300GWM9UJLZCGSN04,R0MUWSFPU8MPRO8K5P83,TAKE,F2KUVLIFOYYYLY7A86QTJNMN76CPND8WN28Z


## 3. How are rows grouped by trade and day?

How many rows share a `tec_ruti` on one day, how many of them carry the best value leg flag, and how many distinct event dates do they have?

In [25]:
query = f"""

SELECT n_rows, n_best, n_dates, COUNT(*) AS n_groups
FROM (
  SELECT tec_ruti, reference_period,
         COUNT(*) AS n_rows,
         SUM(CASE WHEN tec_best_value_leg = 1 THEN 1 ELSE 0 END) AS n_best,
         COUNT(DISTINCT event_date) AS n_dates
  FROM crp_sftds_ecb.trade_states_securitieslending
  GROUP BY tec_ruti, reference_period
) g
GROUP BY n_rows, n_best, n_dates
ORDER BY n_groups DESC

"""
df = pd.read_sql_query(query, cnxn)
df.head(30)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1495388006.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_rows,n_best,n_dates,n_groups
0,1,0,1,121625857
1,2,1,1,16065709
2,2,1,2,4972640
3,1,1,1,1868
4,3,0,2,78
5,126301,0,950,1
6,140510,0,951,1
7,135201,0,953,1
8,137174,0,968,1
9,140785,0,950,1


Answer. Single legs (one row, no flag), pairs (two rows, exactly one flag) and a handful of triples. The best value leg flag is only set within pairs, so it cannot be used as a filter on its own.

In [26]:
df[df['n_rows'] <= 3]

,n_rows,n_best,n_dates,n_groups
0,1,0,1,121625857
1,2,1,1,16065709
2,2,1,2,4972640
3,1,1,1,1868
4,3,0,2,78


What are the lines with more than 100k rows per day? A NULL `tec_ruti` forms one group per day in a `GROUP BY`, so these are the rows without a key. What are they?

In [27]:
query = f"""

SELECT reference_period, action_type,
       CASE WHEN uti IS NULL THEN 1 ELSE 0 END AS uti_missing,
       CASE WHEN loan_security_id IS NULL THEN 1 ELSE 0 END AS isin_missing,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NULL OR tec_ruti = ''
GROUP BY 1, 2, 3, 4
ORDER BY reference_period, n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\3913030310.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,action_type,uti_missing,isin_missing,n
0,2026-06-01,COLU,1,1,129554
1,2026-06-02,COLU,1,1,130317
2,2026-06-03,COLU,1,1,130625
3,2026-06-04,COLU,1,1,132193
4,2026-06-05,COLU,1,1,127244
...,...,...,...,...,...
73,2026-09-10,COLU,1,1,134582
74,2026-09-11,COLU,1,1,133509
75,2026-09-14,COLU,1,1,130676
76,2026-09-15,COLU,1,1,130152


In [28]:
df['action_type'].unique()

array(['COLU'], dtype=object)

In [29]:
df['uti_missing'].unique()

array([1], dtype=int64)

In [30]:
df['isin_missing'].unique()

array([1], dtype=int64)

Answer. Collateral updates on a net exposure basis. They carry no UTI and no ISIN, so they belong to a counterparty pair rather than to a loan, and their event dates are the dates of the last collateral report per pair, which can lie years back. The loan table drops them with `tec_ruti IS NOT NULL`.

Is `tec_surrogate_key` unique per row? It is the join key for the array aggregates.

In [31]:
query = f"""

SELECT COUNT(*), COUNT(DISTINCT tec_surrogate_key)
FROM crp_sftds_ecb.trade_states_securitieslending

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\247840212.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,expr_1,expr_2
0,174315664,174315664


Answer. Yes.

## 4. Where is the collateral?

For which loans do the arrays hold anything? Cross the two flags with whether the arrays are filled.

In [32]:
query = f"""

SELECT collateralisation_net_exposure, uncollateralised_flag,
       CASE WHEN number_collateral_securities > 0 THEN 1 ELSE 0 END AS has_sec,
       CASE WHEN number_collateral_cash > 0 THEN 1 ELSE 0 END AS has_cash,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
GROUP BY 1, 2, 3, 4
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\2575286503.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,collateralisation_net_exposure,uncollateralised_flag,has_sec,has_cash,n
0,True,False,0,0,115704011
1,False,False,0,1,31033129
2,None,True,0,0,10672004
3,False,False,0,0,2216146
4,False,False,1,0,1813612
5,True,False,1,0,1194875
6,True,False,0,1,1038146
7,False,False,1,1,19174
8,None,False,0,0,9917
9,True,False,1,1,3643


Answer. The flag and the arrays describe two layers of collateral. `collateralisation_net_exposure` says that margining runs on the net exposure of the counterparty pair, the arrays say whether this loan also has collateral allocated to it, and SFTR allows both at once (ESMA guidelines, section 5.4.7). So the combinations read as follows.

* Net exposure with empty arrays is the pool only case and the most frequent one. Typical for agency lending, where one collateral pool per borrower and lender pair covers many small loans. The pool sits in the UTI less rows of section 3, the loan row cannot show it.
* Net exposure with cash or securities is a loan with its own collateral that is additionally margined on a net basis, the case in ESMA example 5.4.7.1.
* Both flags false with cash or securities is plain trade level collateral. Cash and securities together is mixed collateral, rare but legitimate.
* Both flags false with empty arrays is basket collateral or missing collateral, see the next query.
* Uncollateralised loans carry nothing by definition. Rows with both flags NULL are incomplete reports.

What about loans with both flags false but neither securities nor cash?

In [33]:
query = f"""

SELECT CASE WHEN collateral_basket_id IS NULL THEN 0 ELSE 1 END AS has_basket, COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
  AND collateralisation_net_exposure = FALSE AND uncollateralised_flag = FALSE
  AND COALESCE(number_collateral_securities, 0) = 0
  AND COALESCE(number_collateral_cash, 0) = 0
GROUP BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\2663623859.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,has_basket,n
0,0,129040
1,1,2087106


Answer. Nearly all of them reference a collateral basket, `collateral_basket_id`. The rest have missing collateral.

How many pieces of securities collateral does a loan carry?

In [34]:
query = f"""

SELECT number_collateral_securities AS n_sec, COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE number_collateral_securities > 0
GROUP BY 1
ORDER BY 1
LIMIT 30

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1949470445.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_sec,n
0,1,3952568
1,2,2528437
2,3,655308
3,4,486641
4,5,397810
5,6,351930
6,7,222543
7,8,166488
8,9,123015
9,10,109055


What does one array element look like? The comma between the table and `t.collateral_security` unnests the array, one row per element.

In [35]:
query = f"""

SELECT t.reference_period, t.uti, e.id, e.market_value_eur, e.haircut_margin, e.security_type, e.quality
FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_security e
WHERE uti IS NOT NULL
LIMIT 10

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1296614772.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,uti,id,market_value_eur,haircut_margin,security_type,quality
0,2026-07-03,F2KUVLIFOYBEVTPMLMYWQ7CAMG8B8N2BF7WZ,CA00208D4084,-6.342470e+05,0.0000,MEQU,NOAP
1,2026-07-03,20260701PAT384394434,ES0000012I32,8.895425e+07,0.0000,GOVS,INVG
2,2026-07-03,20260701PAT384394434,IT0005436693,8.224337e+08,0.0000,GOVS,INVG
3,2026-07-03,F2KUVLIFOYXFY3MGGWUB4A2XEFJGRLKDQZLP,FR0013404969,0.000000e+00,0.0000,GOVS,INVG
4,2026-07-03,F2KUVLIFOY4ZRLY7JT4MEGTGPZR92CNHK6FZ,XS3337424462,1.603152e+07,4.7017,SUNS,INVG
5,2026-07-03,20260701PAT384368839,ES0000012I32,8.895425e+07,0.0000,GOVS,INVG
6,2026-07-03,20260701PAT384368839,IT0005436693,8.224337e+08,0.0000,GOVS,INVG
7,2026-07-03,E02IYKCAVNFR8QGF00HV840SFT2184572,CZ0001007645,3.068259e+07,0.0000,GOVS,NOTR
8,2026-07-03,20260701PAT384392070,ES0000012I32,8.895425e+07,0.0000,GOVS,INVG
9,2026-07-03,20260701PAT384392070,IT0005436693,8.224337e+08,0.0000,GOVS,INVG


In [36]:
query = f"""

SELECT t.reference_period, t.uti, e.amount, e.amount_currency, e.amount_eur, e.haircut_margin
FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_cash e
LIMIT 10

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\2976493077.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,uti,amount,amount_currency,amount_eur,haircut_margin
0,2026-08-06,6PTKHDJ8HDUF78PFWH30SBLXSTKL04823330000110,685023.021,USD,5.944316e+05,102.0
1,2026-08-06,20260804LV60350901210084,2453.200,EUR,2.453200e+03,100.0
2,2026-08-06,BNPBOLIVARY3204433620260623230728,359086.600,EUR,3.590866e+05,0.0
3,2026-08-06,BNPBOLIVARY3194681920260617125833,-1073029.070,EUR,-1.073029e+06,0.0
4,2026-08-06,E02571474TGEMMWANRLN572SFA260708112631547L0342...,784607.850,USD,6.808468e+05,0.0
5,2026-08-06,BNPBOLIVARY3259469320260728074249,139230.000,EUR,1.392300e+05,0.0
6,2026-08-06,BNP549300KUN9K9K32C6D97XFALCON20260805279962B,-350300.000,USD,-3.039743e+05,0.0
7,2026-08-06,BNPBOLIVARY3173116920260604070834,78417.420,EUR,7.841742e+04,0.0
8,2026-08-06,50OBSE5T5521O6SMZR2820260805XIBIESLSL662521308,185.000,USD,1.605345e+02,0.0
9,2026-08-06,F2KUVLIFOY6HBZD7YZRXAJ8WTWXBXZJQ4U4Z,-6506400.000,USD,-5.645956e+06,0.0


Why are market values in the array elements sometimes negative? Cross the sign with the reporting side, separately for loan rows and for the UTI less pool rows.

In [37]:
query = f"""

SELECT CASE WHEN t.tec_ruti IS NULL OR t.tec_ruti = '' THEN 'pool' ELSE 'loan' END AS row_type,
       t.counterparty_side,
       CASE WHEN e.market_value_eur < 0 THEN 'negative'
            WHEN e.market_value_eur > 0 THEN 'positive' ELSE 'zero_or_null' END AS sign,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_security e
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\428617703.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,row_type,counterparty_side,sign,n
0,loan,GIVE,negative,1176021
1,loan,GIVE,positive,4898612
2,loan,GIVE,zero_or_null,15175
3,loan,TAKE,negative,131964
4,loan,TAKE,positive,884579
5,loan,TAKE,zero_or_null,19638
6,pool,None,negative,112926439
7,pool,None,positive,133489682
8,pool,None,zero_or_null,1072493


In [38]:
query = f"""

SELECT CASE WHEN t.tec_ruti IS NULL OR t.tec_ruti = '' THEN 'pool' ELSE 'loan' END AS row_type,
       t.counterparty_side,
       CASE WHEN e.amount_eur < 0 THEN 'negative'
            WHEN e.amount_eur > 0 THEN 'positive' ELSE 'zero_or_null' END AS sign,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_cash e
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\2885550702.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,row_type,counterparty_side,sign,n
0,loan,GIVE,negative,15607825
1,loan,GIVE,positive,7435969
2,loan,GIVE,zero_or_null,254477
3,loan,TAKE,negative,3725016
4,loan,TAKE,positive,4999289
5,loan,TAKE,zero_or_null,86668
6,pool,None,negative,200024
7,pool,None,positive,379182
8,pool,None,zero_or_null,97303


Answer. There is no clean convention on the loan rows. Negative values appear on both sides, for cash on 68 percent of GIVE elements and 43 percent of TAKE elements, for securities on 19 and 13 percent. Some reporters apply the perspective sign to trade level collateral, others do not, so the sign carries no reliable direction and the cleaning query should aggregate absolute values. The pool rows carry no `counterparty_side`, as ESMA prescribes for net exposure collateral, so there the sign is the only indicator of direction, negative for the provider and positive for the taker.

## 5. How is the price of the loan stored?

Which rate columns are filled for fixed rebates, floating rebates and fee based loans?

In [39]:
query = f"""

SELECT rebate_rate_type, COUNT(*) AS n,
       COUNT(fxd_rebate_rate) AS n_fixed_rate,
       COUNT(flt_rebate_rate) AS n_float_index,
       COUNT(rebate_rate_derived_sdw) AS n_derived_rate,
       COUNT(lending_fee) AS n_fee
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
GROUP BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1725497723.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,rebate_rate_type,n,n_fixed_rate,n_float_index,n_derived_rate,n_fee
0,Fixed,29605129,29605129,0,29605129,2735680
1,None,131360125,0,0,0,128320407
2,Floating,2739403,0,1180844,962822,234369


Is `rebate_rate_derived_sdw` the floating index plus the spread?

In [40]:
query = f"""

SELECT rebate_rate_type, COUNT(*) AS n,
       APPX_MEDIAN(rebate_rate_derived_sdw
                   - (flt_rebate_rate_value_sdw + flt_rebate_rate_spread_basispoints / 100)) AS med_diff,
       MIN(rebate_rate_derived_sdw) AS min_rate, MAX(rebate_rate_derived_sdw) AS max_rate
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE flt_rebate_rate IS NOT NULL
GROUP BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\614880981.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,rebate_rate_type,n,med_diff,min_rate,max_rate
0,Floating,1180844,0.0,3.3,11.56


Answer. Yes, the median difference is zero, and rates are in percent per annum.

The cleaning query built from these checks is in `sec_lending_clean_query.txt`.